# Fetch API 
- No API-key requiered

In [21]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
CONTACT = os.getenv("MB_CONTACT")

url = "https://musicbrainz.org/ws/2/recording"
headers = {"User-Agent": f"MusikQuizApp/0.1 ({CONTACT})"}
params = {
    "query": "tag:rock AND date:[1980 TO 1989]",
    "limit": 25,
    "fmt": "json",
}

r = requests.get(url, params=params, headers=headers, timeout=30)
r.raise_for_status()

df = pd.json_normalize(r.json()["recordings"])
df.columns

Index(['id', 'score', 'artist-credit-id', 'title', 'length', 'disambiguation',
       'video', 'artist-credit', 'first-release-date', 'releases', 'isrcs',
       'tags'],
      dtype='str')

### Fetch data using function

In [22]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads in .env

BASE_URL = "https://musicbrainz.org/ws/2"
CONTACT = os.getenv("MB_CONTACT")
if not CONTACT:
    raise RuntimeError("MB_CONTACT missing in .env")

HEADERS = {"User-Agent": f"MusikQuizApp/0.1 ({CONTACT})"}

def mb_get(endpoint, params):
    params = {**params, "fmt": "json"}
    r = requests.get(f"{BASE_URL}/{endpoint}", params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    time.sleep(1)  # rate limit
    return r.json()

# Ex: recordnings from the 80's by a certain artist
data = mb_get("recording", {
    "query": 'artist:"a-ha" AND date:[1980 TO 1989]',
    "limit": 100,
    "offset": 0,
})

print(data["count"])
print(data["recordings"][0])

100
{'id': '42f98955-ed53-48d3-8c31-578f1c63d935', 'score': 100, 'artist-credit-id': '07885c4d-864e-30be-ba2b-541c03409c3b', 'title': 'Manhattan Skyline', 'length': 290000, 'video': None, 'artist-credit': [{'name': 'a‐ha', 'artist': {'id': '7364dea6-ca9a-48e3-be01-b44ad0d19897', 'name': 'a‐ha', 'sort-name': 'a‐ha', 'disambiguation': 'Norwegian synth‐pop band', 'aliases': [{'sort-name': 'A-ha', 'name': 'A-ha', 'locale': None, 'type': None, 'primary': None, 'begin-date': None, 'end-date': None}, {'sort-name': 'A Ha', 'name': 'A Ha', 'locale': None, 'type': None, 'primary': None, 'begin-date': None, 'end-date': None}, {'sort-name': 'Aha', 'name': 'Aha', 'locale': None, 'type': None, 'primary': None, 'begin-date': None, 'end-date': None}]}}], 'first-release-date': '1986-11-27', 'releases': [{'id': '2d147ea1-d1b6-4a2c-b117-94f7cafcb049', 'status-id': '4e304316-386d-3409-af2e-78857eec5cfe', 'artist-credit-id': '07885c4d-864e-30be-ba2b-541c03409c3b', 'count': 1, 'title': 'Scoundrel Days', 'st

### Some EDA artist "a-ha" between dates 1980 - 1989

In [23]:
# Check columns 
df = pd.json_normalize(data["recordings"])
df.columns

Index(['id', 'score', 'artist-credit-id', 'title', 'length', 'video',
       'artist-credit', 'first-release-date', 'releases', 'isrcs', 'tags',
       'disambiguation'],
      dtype='str')

In [24]:
# first rows
df.head()

,id,score,artist-credit-id,title,length,video,artist-credit,first-release-date,releases,isrcs,tags,disambiguation
0,42f98955-ed53-48d3-8c31-578f1c63d935,100,07885c4d-864e-30be-ba2b-541c03409c3b,Manhattan Skyline,290000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1986-11-27,[{'id': '2d147ea1-d1b6-4a2c-b117-94f7cafcb049'...,[USWB10107602],"[{'count': 1, 'name': 'rock'}, {'count': 2, 'n...",NaN
1,b21ec767-1814-4107-8673-7805906d6378,100,07885c4d-864e-30be-ba2b-541c03409c3b,There's Never a Forever Thing,168000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1988,[{'id': '93d73efa-866e-461a-bd6e-152163bb2a70'...,NaN,NaN,NaN
2,944acd96-072f-4714-bf9b-d8a1e9189d35,100,07885c4d-864e-30be-ba2b-541c03409c3b,We’re Looking for the Whales (live version),412000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1987-05-25,[{'id': '153bc4f7-f77c-493b-a580-12d860d7fca4'...,NaN,NaN,NaN
3,1002235f-43ec-4c1f-abdb-a7ecc9f90f44,100,07885c4d-864e-30be-ba2b-541c03409c3b,Take On Me,227000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1989,[{'id': 'df0e75ae-4ac5-4d35-9bab-f8b4deda6868'...,NaN,NaN,live
4,2f97a4d3-8ea0-4dff-8d70-0921fe8bc609,100,07885c4d-864e-30be-ba2b-541c03409c3b,The Sun Always Shines On T.V. (instrumental),400000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1985,[{'id': '739f5e52-36d7-489f-8055-f4ca54b4733b'...,NaN,NaN,NaN


In [25]:
# types and number of missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  100 non-null    str    
 1   score               100 non-null    int64  
 2   artist-credit-id    100 non-null    str    
 3   title               100 non-null    str    
 4   length              95 non-null     float64
 5   video               0 non-null      object 
 6   artist-credit       100 non-null    object 
 7   first-release-date  100 non-null    str    
 8   releases            100 non-null    object 
 9   isrcs               2 non-null      object 
 10  tags                7 non-null      object 
 11  disambiguation      11 non-null     str    
dtypes: float64(1), int64(1), object(5), str(5)
memory usage: 19.6+ KB


In [26]:
# number of rows and columns
df.shape

(100, 12)

### Check with another artist and years

In [27]:
data = mb_get("recording", {
    "query": 'artist:"Britney Spears" AND date:[1995 TO 2005]',
    "limit": 100,
    "offset": 0,
})

print(data["count"])
print(data["recordings"][0])

488
{'id': 'b89f48bb-5dce-4ce3-aec9-364477f526fa', 'score': 100, 'artist-credit-id': '038401cc-de8b-3013-9efe-4ee3e07782f6', 'title': 'Boys', 'length': 242000, 'disambiguation': 'part of “The Very Best of Pure R&B: The Summer Collection” DJ‐mix', 'video': None, 'artist-credit': [{'name': 'Britney Spears', 'artist': {'id': '45a663b5-b1cb-4a91-bff6-2bef7bbfdd76', 'name': 'Britney Spears', 'sort-name': 'Spears, Britney', 'aliases': [{'sort-name': '브리트니 스피어스', 'type-id': '1937e404-b981-3cb7-8151-4c86ebfc8d8e', 'name': '브리트니 스피어스', 'locale': None, 'type': 'Search hint', 'primary': None, 'begin-date': None, 'end-date': None}, {'sort-name': 'ブリトニー・スピアーズ', 'type-id': '894afba6-2816-3c24-8072-eadb66bd04bc', 'name': 'ブリトニー・スピアーズ', 'locale': 'ja', 'type': 'Artist name', 'primary': True, 'begin-date': None, 'end-date': None}, {'sort-name': 'Bretney Spears', 'name': 'Bretney Spears', 'locale': None, 'type': None, 'primary': None, 'begin-date': None, 'end-date': None}, {'sort-name': 'B. Spears', 'na

In [28]:
df = pd.json_normalize(data["recordings"])
df.columns

Index(['id', 'score', 'artist-credit-id', 'title', 'length', 'disambiguation',
       'video', 'artist-credit', 'first-release-date', 'releases', 'isrcs',
       'tags'],
      dtype='str')

In [29]:
df.head()

,id,score,artist-credit-id,title,length,disambiguation,video,artist-credit,first-release-date,releases,isrcs,tags
0,b89f48bb-5dce-4ce3-aec9-364477f526fa,100,038401cc-de8b-3013-9efe-4ee3e07782f6,Boys,242000.0,part of “The Very Best of Pure R&B: The Summer...,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",2003-03-31,[{'id': 'dfeca7a0-f9c8-48e3-b609-24debb4769be'...,NaN,NaN
1,5286dc3a-c699-4784-8bde-12701932f0fd,100,038401cc-de8b-3013-9efe-4ee3e07782f6,(You Drive Me) Crazy (The Stop Remix!) (instru...,199400.0,NaN,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",1999-09-29,[{'id': '9eb6afce-a06c-468c-8bca-1408a8881f0f'...,[USJI19910720],"[{'count': 1, 'name': 'electronic'}, {'count':..."
2,fb975c61-c934-4880-835e-d23437401b25,100,038401cc-de8b-3013-9efe-4ee3e07782f6,(You Drive Me) Crazy (Jazzy Jim’s hip-hop mix),220000.0,NaN,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",1999-12-06,[{'id': 'acaee915-0ecd-4e3a-a112-be1d2e6d9641'...,NaN,"[{'count': 3, 'name': 'pop'}, {'count': 1, 'na..."
3,d9a51139-373d-48ab-9539-2af59860a2b3,100,038401cc-de8b-3013-9efe-4ee3e07782f6,Right Now (Taste the Victory),314760.0,NaN,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",2002-04,[{'id': '54593f6e-5251-439c-a45b-102b940773da'...,NaN,NaN
4,2c4e741b-b0d0-452a-994e-226925516632,100,038401cc-de8b-3013-9efe-4ee3e07782f6,Toxic,186000.0,NaN,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",2004,[{'id': '3ef19145-4c5a-4ede-8d0f-1c9a4c3c225b'...,NaN,NaN


### Test with 'genre' and 'relese'

In [32]:
data = mb_get("recording", {
    "query": 'genre:"pop" AND relese:[2005]',
    "limit": 100,
    "offset": 0,
})
print(data)

{'created': '2026-10-12T09:16:49.401Z', 'count': 7715477, 'offset': 0, 'recordings': [{'id': 'd2fd57de-995c-41d4-be13-179d3ca7b37c', 'score': 100, 'artist-credit-id': 'a4ea3a21-77ec-3f61-aed9-56e440b9f043', 'title': 'Pop Sub Genre', 'length': 373136, 'video': None, 'artist-credit': [{'name': 'Daffodill', 'artist': {'id': 'a1d48753-eb5e-40cf-a881-27163c0dd264', 'name': 'Daffodill', 'sort-name': 'Daffodill', 'disambiguation': 'Stephen Davies'}}], 'first-release-date': '2024-04-25', 'releases': [{'id': '4736c140-77ae-47a8-9d70-10db8d801eea', 'status-id': '4e304316-386d-3409-af2e-78857eec5cfe', 'count': 1, 'title': 'CHASTITY, 159', 'status': 'Official', 'release-group': {'id': '4e985c85-c302-470f-9e4f-1dd8d1deec7e', 'type-id': 'f529b476-6e62-324f-b0aa-1f3e33d313fc', 'primary-type-id': 'f529b476-6e62-324f-b0aa-1f3e33d313fc', 'title': 'CHASTITY, 159', 'primary-type': 'Album'}, 'date': '2024-04-25', 'country': 'XW', 'release-events': [{'date': '2024-04-25', 'area': {'id': '525d4e18-3d00-31b9-

In [34]:
df = pd.json_normalize(data["recordings"])
df

,id,score,artist-credit-id,title,length,video,artist-credit,first-release-date,releases,isrcs,tags,disambiguation
0,d2fd57de-995c-41d4-be13-179d3ca7b37c,100,a4ea3a21-77ec-3f61-aed9-56e440b9f043,Pop Sub Genre,373136.0,None,"[{'name': 'Daffodill', 'artist': {'id': 'a1d48...",2024-04-25,[{'id': '4736c140-77ae-47a8-9d70-10db8d801eea'...,NaN,NaN,NaN
1,fa427675-f624-4fb4-94ff-2e0487eeeb80,91,0519cbf0-f286-3cf7-96da-8f0f13478567,Silver,NaN,None,"[{'name': 'Relese', 'artist': {'id': '34cf8d4a...",2001-03-11,[{'id': '9230145a-2f26-4f70-bc0a-18a97c0b403d'...,NaN,NaN,NaN
2,d5c6799a-8fa2-41d8-8e8e-e8e74a193ff4,82,701b6f2c-e323-4440-98a0-ea4c25e95697,Popular Music Genre,60330.0,None,"[{'name': 'Arcangel Legrand - Brotherswagg', '...",2027-12-13,[{'id': 'ce1e61ac-c76e-429f-9d41-d15f055876e2'...,NaN,NaN,NaN
3,200e5f3b-3681-4351-abeb-75c642149543,80,0f62e159-a9ad-3cac-96dd-59326497b18c,Multiple Genre Genre,NaN,None,"[{'name': 'Champion the Underdog', 'artist': {...",2014-06-06,[{'id': 'a5de97db-758d-4924-9093-a346288d393d'...,[TCABW1482324],NaN,NaN
4,31ae84bf-a0ec-413c-9c8f-824e73b9415f,79,2673e1ad-09df-30c7-b730-2e87f720d10e,Genres - Pop,107933.0,None,"[{'name': 'Brett Manning', 'artist': {'id': 'c...",NaN,[{'id': '3a21ff1f-0173-4560-83ed-23297e34ec35'...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
95,e9509129-b3f5-4334-ae4c-794a02b4ba5b,67,1bfcfdc6-4f5a-33b9-9d15-5076fadd0330,Genre Bitches,434000.0,None,"[{'name': 'Fagin’s Reject', 'artist': {'id': '...",2014-06-27,[{'id': 'a4799ed4-14c4-4740-9c1d-8af02df28ba6'...,NaN,NaN,NaN
96,5799d903-79f4-4ca9-82e6-91a8044326f2,67,6c13a5ce-eb2e-3502-8737-e38722133512,New Genre,231325.0,None,"[{'name': 'Danny Matos', 'artist': {'id': '77b...",2016-01-08,[{'id': 'df73cfaf-296c-4333-a609-26ec48c39a00'...,NaN,NaN,NaN
97,21d16f12-ded6-4323-afbb-b2df464872e1,67,1dd65c35-c379-3231-b72b-b88ec39849a9,My Genre,NaN,None,"[{'name': 'Aspects', 'artist': {'id': '602dd07...",2002,[{'id': '7f7d6136-2b44-48b7-80e9-9d3b7a4465e7'...,NaN,NaN,NaN
98,682848d9-baed-4a32-b256-ad19eced5548,67,482b9f0d-39a8-31f1-b68e-ba196b912827,Mauvais Genre,167533.0,None,"[{'name': 'Apolline', 'artist': {'id': '11aee4...",2013-01-01,[{'id': '07af4bc1-b383-4550-88dc-c0c8534be9e9'...,[FRP621566230],NaN,NaN


## Check posibility for "url"  to use for spotify (to play records)
- result 100/100 is missing
- 0.09 % has isrc - too few

In [31]:
# 1. Hämta ett urval via sökning (använder mb_get från tidigare)
data = mb_get("recording", {
    "query": "firstreleasedate:[1990 TO 2000] AND tag:rock",
    "limit": 100,
})
recs = data["recordings"]

# 2. Slå upp url-relationer för varje låt
def spotify_urls(relations):
    return [
        rel["url"]["resource"]
        for rel in relations
        if "open.spotify.com" in rel.get("url", {}).get("resource", "")
    ]

rows = []
for rec in recs:
    full = mb_get(f"recording/{rec['id']}", {"inc": "url-rels isrcs"})
    urls = spotify_urls(full.get("relations", []))
    rows.append({
        "mbid": rec["id"],
        "title": rec["title"],
        "has_spotify_url": len(urls) > 0,
        "spotify_urls": urls,
        "has_isrc": len(full.get("isrcs", [])) > 0,
    })

df_check = pd.DataFrame(rows)

# 3. Hur många saknar?
print(df_check["has_spotify_url"].value_counts())
print(df_check["has_spotify_url"].mean())   # andel som har
print(df_check["has_isrc"].mean())          # andel med ISRC

has_spotify_url
False    100
Name: count, dtype: int64
0.0
0.09
